! pip install playwright
! playwright install chromium

In [ ]:
import asyncio
import re
import time
from playwright.async_api import async_playwright



async def attach_to_existing_chrome():
    p = await async_playwright().start()
    browser = await p.chromium.connect_over_cdp("http://localhost:9222")
    if not browser.contexts:
        raise RuntimeError("没有拿到任何 browser context。请确认 Chrome 是用 --remote-debugging-port=9222 启动的。")
    context = browser.contexts[0]

    game_page = None
    for pg in context.pages:
        if "test.milkywayidle.com/game" in pg.url:
            game_page = pg
            break

    if game_page is None:
        game_page = await context.new_page()
        await game_page.goto(GAME_URL, wait_until="domcontentloaded")

    return p, browser, context, game_page


async def wait_for_checkout_page(context, known_pages, timeout=15):
    end = time.time() + timeout
    while time.time() < end:
        for pg in context.pages:
            if pg not in known_pages and "checkout.stripe.com" in pg.url:
                await pg.wait_for_load_state("domcontentloaded")
                return pg
        await asyncio.sleep(0.2)

    for pg in context.pages:
        if "checkout.stripe.com" in pg.url:
            return pg

    raise RuntimeError("没有捕获到 Stripe checkout 页面。")


import random
from pathlib import Path

DEBUG_DIR = Path("autopay_debug")
DEBUG_DIR.mkdir(exist_ok=True)


async def wait_for_store_ready(game_page, timeout_ms=15000):
    store_btn = game_page.get_by_text(re.compile(r"^Cowbell Store$", re.I)).first
    buy_tab = game_page.get_by_text(re.compile(r"^Buy Cowbells$", re.I)).first

    await store_btn.wait_for(state="visible", timeout=timeout_ms)
    await store_btn.click()

    await buy_tab.wait_for(state="visible", timeout=timeout_ms)
    await buy_tab.click()

    pack = await first_visible_locator([
        game_page.get_by_text(re.compile(r"11500\s+Cowbells", re.I)),
        game_page.get_by_text(re.compile(r"Price:\s*\$100", re.I)),
    ], timeout=max(8, timeout_ms / 1000))

    if pack is None:
        raise RuntimeError("已进入 Cowbell Store，但 $100 包仍未出现。")

    return pack


async def open_100_dollar_package(game_page, game_url=None, max_attempts=3):
    last_error = None

    for attempt in range(1, max_attempts + 1):
        try:
            if game_url is not None:
                await game_page.goto(game_url, wait_until="domcontentloaded")

            # 给前端一点渲染时间，并处理登录后偶发弹窗
            await game_page.wait_for_timeout(1000 + 300 * attempt)
            await maybe_close_post_login_dialog(game_page, timeout_ms=2500)

            # 进入商店
            region = await enter_store_and_get_region(game_page, timeout_ms=15000)

            # 再清一次可能遮挡商店的弹窗
            # await dismiss_any_open_modal(game_page)

            # 只在商店区域里找 $100 包
            package_el = await wait_for_100_package_in_region(region, timeout_sec=10 + attempt * 2)
            if package_el is None:
                raise RuntimeError("已进入商店区域，但 $100 包未出现。")

            await package_el.click()
            # print(f"已点击 $100 包（attempt {attempt}/{max_attempts}）")

            await game_page.wait_for_timeout(700)

            # 继续处理确认弹窗
            continue_btn = game_page.get_by_role("button", name=re.compile(r"^Continue$", re.I)).first
            modal_title = game_page.get_by_text(re.compile(r"Continue to Purchase", re.I)).first

            await modal_title.wait_for(state="visible", timeout=8000)
            await continue_btn.wait_for(state="visible", timeout=8000)
            # print("检测到 Continue to Purchase 弹窗")

            # 拦截 window.open，避免真实新标签页抢前台
            await install_window_open_capture(game_page)
            await continue_btn.click()
            checkout_url = await wait_for_checkout_url(game_page, timeout_ms=4000)

            if not checkout_url or "checkout.stripe.com" not in checkout_url:
                raise RuntimeError(f"没有成功捕获 checkout URL: {checkout_url}")

            # print("已捕获 checkout URL")

            checkout_page = await new_checkout_worker_tab(game_page.context)
            await checkout_page.goto(checkout_url, wait_until="domcontentloaded")
            return checkout_page

        except Exception as e:
            last_error = e
            print(f"open_100_dollar_package attempt {attempt}/{max_attempts} failed: {e}")

            try:
                await game_page.screenshot(
                    path=str(DEBUG_DIR / f"open_pkg_fail_attempt_{attempt}.png"),
                    full_page=True
                )
            except:
                pass

            if attempt < max_attempts:
                try:
                    await game_page.reload(wait_until="domcontentloaded")
                except:
                    if game_url is not None:
                        await game_page.goto(game_url, wait_until="domcontentloaded")

                await game_page.wait_for_timeout(1200 + 500 * attempt)

    raise RuntimeError(f"连 $100 包都没点到。最后错误: {last_error}")


async def first_visible_locator(locators, timeout=5):
    end = time.time() + timeout
    while time.time() < end:
        for loc in locators:
            try:
                if await loc.count() > 0 and await loc.first.is_visible():
                    return loc.first
            except:
                pass
        await asyncio.sleep(0.2)
    return None


async def set_quantity_on_checkout(checkout_page, target=99):
    qty_btn = checkout_page.get_by_role("button", name=re.compile(r"^Qty\s+\d+$")).first
    await qty_btn.wait_for(state="visible", timeout=10000)
    # print("当前数量按钮文本:", await qty_btn.inner_text())

    await qty_btn.click()
    await checkout_page.wait_for_timeout(500)

    qty_input = await first_visible_locator([
        checkout_page.get_by_role("dialog").locator("input"),
        checkout_page.locator("[aria-modal='true'] input"),
        checkout_page.locator("input:focus"),
    ], timeout=5)

    if qty_input is None:
        # raise RuntimeError("点开 Qty 后没找到弹窗里的数量输入框。")

        await qty_input.click()
        try:
            await qty_input.press("Meta+A")   # macOS
        except:
            pass
    await qty_input.fill(str(target))

    update_btn = checkout_page.get_by_role("button", name=re.compile(r"^Update$", re.I)).first
    await update_btn.click()
    await checkout_page.wait_for_timeout(800)

    # print("更新后数量按钮文本:", await qty_btn.inner_text())


async def main_attach_only():
    p, browser, context, game_page = await attach_to_existing_chrome()
    return p, browser, context, game_page

async def install_window_open_capture(page):
    await page.evaluate("""
    () => {
        if (window.__pw_open_capture_installed) return;

        window.__pw_open_capture_installed = true;
        window.__pw_last_open_url = null;

        const originalOpen = window.open.bind(window);
        window.__pw_original_open = originalOpen;

        window.open = function(url, ...args) {
            window.__pw_last_open_url = url;
            return null;   // 阻止真实新标签页弹出
        };
    }
    """)


async def new_checkout_worker_tab(context):
    pg = await context.new_page()
    await pg.goto("about:blank")
    return pg


p, browser, context, game_page = await main_attach_only()

checkout_page = await open_100_dollar_package(game_page)
await set_quantity_on_checkout(checkout_page, 99)

In [ ]:
import re
import asyncio
import time
from dataclasses import dataclass

TEST_CARD   = "4242424242424242"
TEST_EXP    = "1234"      # 12/34
TEST_CVC    = "123"
TEST_NAME   = "Test User"
TEST_POSTAL = "10001"


@dataclass
class FastManualFields:
    pay_without_link: object = None
    submit: object = None
    card_number: object = None
    expiry: object = None
    cvc: object = None
    cardholder: object = None
    postal: object = None


async def _first_visible(candidates, timeout_ms=2500):
    end = time.time() + timeout_ms / 1000
    while time.time() < end:
        for loc in candidates:
            try:
                cand = loc.first
                if await cand.count() > 0 and await cand.is_visible():
                    return cand
            except Exception:
                pass
        await asyncio.sleep(0.12)
    return None

async def _clear_and_fill(locator, value: str):
    await locator.click()
    try:
        await locator.press("Meta+A")      # macOS
    except Exception:
        pass
    try:
        await locator.press("Control+A")   # 兜底
    except Exception:
        pass
    await locator.fill(value)


async def _fill_if_needed(locator, value: str, skip_if_nonempty=True):
    if locator is None:
        return False

    try:
        current = await locator.input_value()
    except Exception:
        current = ""

    if skip_if_nonempty and current.strip():
        return True

    await _clear_and_fill(locator, value)
    return True


async def wait_for_payment_done_and_close(checkout_page, game_page, timeout_ms=30000, refresh_game=False):
    try:
        await checkout_page.wait_for_url("**/paymentSuccess", timeout=timeout_ms)
        # print("检测到已跳转到 paymentSuccess")
    except Exception:
        # 再退回文本兜底
        success_patterns = [
            re.compile(r"Payment successful", re.I),
            re.compile(r"Thanks|Thank you", re.I),
            re.compile(r"Complete|Completed", re.I),
            re.compile(r"Success", re.I),
        ]
        hit = False
        for pat in success_patterns:
            try:
                loc = checkout_page.get_by_text(pat).first
                await loc.wait_for(state="visible", timeout=3000)
                # print("检测到支付完成提示：", pat.pattern)
                hit = True
                break
            except:
                pass
        if not hit:
            raise RuntimeError(f"等待支付完成超时。当前 URL: {checkout_page.url}")

    await checkout_page.close()
    if refresh_game:
        await game_page.goto(game_page.url, wait_until="domcontentloaded")


    
    
async def _resolve_on_target(target, kind, timeout_ms=1500):
    if kind == "pay_without_link":
        return await _first_visible([
            target.get_by_role("button", name=re.compile(r"Pay without Link", re.I)),
            target.get_by_text(re.compile(r"Pay without Link", re.I)),
        ], timeout_ms=timeout_ms)

    if kind == "submit":
        return await _first_visible([
            target.locator("button[type='submit']"),
            target.get_by_role("button", name=re.compile(r"^(Pay|Complete|Confirm|Subscribe)", re.I)),
        ], timeout_ms=timeout_ms)

    if kind == "card_number":
        return await _first_visible([
            target.get_by_label(re.compile(r"Card number", re.I)),
            target.get_by_placeholder(re.compile(r"1234 1234 1234 1234", re.I)),
            target.locator('input[autocomplete="cc-number"]'),
            target.locator('input[name*="cardnumber"]'),
        ], timeout_ms=timeout_ms)

    if kind == "expiry":
        return await _first_visible([
            target.get_by_label(re.compile(r"MM\s*/\s*YY|Exp(?:iry|iration)", re.I)),
            target.get_by_placeholder(re.compile(r"MM\s*/\s*YY", re.I)),
            target.locator('input[autocomplete="cc-exp"]'),
            target.locator('input[name*="exp"]'),
        ], timeout_ms=timeout_ms)

    if kind == "cvc":
        return await _first_visible([
            target.get_by_label(re.compile(r"CVC|CVV|Security code", re.I)),
            target.get_by_placeholder(re.compile(r"CVC|CVV", re.I)),
            target.locator('input[autocomplete="cc-csc"]'),
            target.locator('input[name*="cvc"], input[name*="cvv"]'),
        ], timeout_ms=timeout_ms)

    if kind == "cardholder":
        return await _first_visible([
            target.get_by_label(re.compile(r"Cardholder name|Name on card", re.I)),
            target.get_by_placeholder(re.compile(r"Cardholder name|Name on card", re.I)),
            target.locator('input[autocomplete="cc-name"]'),
        ], timeout_ms=timeout_ms)

    if kind == "postal":
        return await _first_visible([
            target.get_by_label(re.compile(r"ZIP|Postal code|Postcode", re.I)),
            target.get_by_placeholder(re.compile(r"ZIP|Postal code|Postcode", re.I)),
            target.locator('input[autocomplete="postal-code"]'),
        ], timeout_ms=timeout_ms)

    return None


async def resolve_manual_fields_fast(checkout_page):
    """
    针对你当前这条固定路径做的快速版：
    先点 Pay without Link，然后优先在主页面直接抓字段。
    只有主页面抓不到时，才对 frames 做一次很短的兜底。
    """
    f = FastManualFields()

    # await checkout_page.bring_to_front()
    await checkout_page.wait_for_timeout(400)

    # 先处理 Pay without Link
    f.pay_without_link = await _resolve_on_target(checkout_page, "pay_without_link", timeout_ms=1200)
    if f.pay_without_link is not None:
        # print("点击 Pay without Link")
        await f.pay_without_link.click()
        await checkout_page.wait_for_timeout(500)

    # 先在主页面等 card number，主页面是最快路径
    f.card_number = await _resolve_on_target(checkout_page, "card_number", timeout_ms=5000)

    if f.card_number is not None:
        f.expiry     = await _resolve_on_target(checkout_page, "expiry", timeout_ms=800)
        f.cvc        = await _resolve_on_target(checkout_page, "cvc", timeout_ms=800)
        f.cardholder = await _resolve_on_target(checkout_page, "cardholder", timeout_ms=800)
        f.postal     = await _resolve_on_target(checkout_page, "postal", timeout_ms=800)
    else:
        # 主页面抓不到，才做一次短兜底
        for frame in checkout_page.frames:
            if frame == checkout_page.main_frame:
                continue

            if f.card_number is None:
                f.card_number = await _resolve_on_target(frame, "card_number", timeout_ms=800)
            if f.expiry is None:
                f.expiry = await _resolve_on_target(frame, "expiry", timeout_ms=500)
            if f.cvc is None:
                f.cvc = await _resolve_on_target(frame, "cvc", timeout_ms=500)
            if f.cardholder is None:
                f.cardholder = await _resolve_on_target(frame, "cardholder", timeout_ms=500)
            if f.postal is None:
                f.postal = await _resolve_on_target(frame, "postal", timeout_ms=500)

        # 再补一次主页面姓名/邮编
        if f.cardholder is None:
            f.cardholder = await _resolve_on_target(checkout_page, "cardholder", timeout_ms=500)
        if f.postal is None:
            f.postal = await _resolve_on_target(checkout_page, "postal", timeout_ms=500)

    f.submit = await _resolve_on_target(checkout_page, "submit", timeout_ms=1500)

    # print("fast manual fields:", {
    #     "submit": f.submit is not None,
    #     "card_number": f.card_number is not None,
    #     "expiry": f.expiry is not None,
    #     "cvc": f.cvc is not None,
    #     "cardholder": f.cardholder is not None,
    #     "postal": f.postal is not None,
    # })

    return f


async def fill_manual_fields_fast(fields):
    if fields.card_number is None:
        raise RuntimeError("快速路径下仍未定位到 card number。")

    await _clear_and_fill(fields.card_number, TEST_CARD)

    if fields.expiry is not None:
        await _clear_and_fill(fields.expiry, TEST_EXP)

    if fields.cvc is not None:
        await _clear_and_fill(fields.cvc, TEST_CVC)

    if fields.cardholder is not None:
        await _fill_if_needed(fields.cardholder, TEST_NAME, skip_if_nonempty=True)

    if fields.postal is not None:
        await _fill_if_needed(fields.postal, TEST_POSTAL, skip_if_nonempty=True)


async def submit_checkout_only_fast(checkout_page):
    fields = await resolve_manual_fields_fast(checkout_page)
    await fill_manual_fields_fast(fields)

    if fields.submit is None:
        raise RuntimeError("没有找到支付按钮。")

    await fields.submit.click()

    # 给前端一个很短的时间进入 processing 状态
    await checkout_page.wait_for_timeout(200)

    return {
        "page": checkout_page,
        "submitted_at": time.time(),
    }


SUCCESS_PATTERNS = [
    re.compile(r"Payment successful", re.I),
    re.compile(r"Thanks|Thank you", re.I),
    re.compile(r"Complete|Completed", re.I),
    re.compile(r"Success", re.I),
]

async def checkout_is_done(page):
    if page.is_closed():
        return True

    try:
        url = page.url
    except:
        return True

    # 最稳的完成信号
    if "paymentSuccess" in url:
        return True

    # 离开 Stripe 也可视为完成/已跳走
    if "checkout.stripe.com" not in url:
        return True

    # 文本兜底，不阻塞等待
    for pat in SUCCESS_PATTERNS:
        try:
            loc = page.get_by_text(pat).first
            if await loc.count() > 0 and await loc.is_visible():
                return True
        except:
            pass

    return False


async def reap_finished_checkouts(pending_checkouts):
    still_pending = []
    finished = 0

    for item in pending_checkouts:
        page = item["page"]
        done = False
        try:
            done = await checkout_is_done(page)
        except:
            done = True

        if done:
            try:
                if not page.is_closed():
                    await page.close()
            except:
                pass
            finished += 1
        else:
            still_pending.append(item)

    return still_pending, finished


MAX_INFLIGHT_PER_ACCOUNT = 2   # 先从 2 开始；稳了再试 3

async def wait_for_checkout_slot(pending_checkouts, max_inflight=MAX_INFLIGHT_PER_ACCOUNT):
    while len(pending_checkouts) >= max_inflight:
        pending_checkouts, finished = await reap_finished_checkouts(pending_checkouts)
        if finished == 0:
            await asyncio.sleep(0.3)
    return pending_checkouts

# async def submit_and_return_fast(checkout_page, game_page, refresh_game=False):
#     fields = await resolve_manual_fields_fast(checkout_page)
#     await fill_manual_fields_fast(fields)

#     # await checkout_page.screenshot(path="05_manual_ready_fast.png", full_page=True)

#     if fields.submit is None:
#         raise RuntimeError("没有找到支付按钮。")

#     try:
#         btn_text = await fields.submit.inner_text()
#     except Exception:
#         btn_text = "<no text>"

#     # print("点击支付按钮：", btn_text)
#     await fields.submit.click()

#     await wait_for_payment_done_and_close(
#         checkout_page,
#         game_page,
#         timeout_ms=60000,
#         refresh_game=refresh_game,
#     )

#     # print("已关闭支付页，已回到游戏页")
#     return "manual-fast"

async def connect_headless():
    p = await async_playwright().start()
    browser = await p.chromium.launch(
        channel="chromium",   # 新 headless，更接近真实 Chrome
        headless=True,
    )
    context = await browser.new_context(
        storage_state="mwidle_state.json"
    )
    game_page = await context.new_page()
    await game_page.goto(GAME_URL, wait_until="domcontentloaded")
    return p, browser, context, game_page


async def maybe_close_post_login_dialog(game_page, timeout_ms=1200):
    # 先快速判断是否真有弹窗
    dialog = await first_visible_locator([
        game_page.locator("[role='dialog']"),
        game_page.locator("[aria-modal='true']"),
        game_page.get_by_role("dialog"),
    ], timeout=0.4)

    if dialog is None:
        return False

    patterns = [
        re.compile(r"close|关闭|关 闭", re.I),
        re.compile(r"ok|确定|知道了|got it", re.I),
        re.compile(r"dismiss|skip", re.I),
        re.compile(r"×|✕|✖|x", re.I),
    ]

    for pat in patterns:
        btn = await first_visible_locator([
            dialog.get_by_role("button", name=pat),
            dialog.get_by_text(pat),
        ], timeout=0.3)
        if btn is not None:
            try:
                await btn.click()
                return True
            except:
                pass

    try:
        await game_page.keyboard.press("Escape")
        await game_page.wait_for_timeout(100)
        return True
    except:
        pass

    try:
        box = await game_page.locator("body").bounding_box()
        if box:
            await game_page.mouse.click(box["x"] + 20, box["y"] + 20)
            await game_page.wait_for_timeout(100)
            return True
    except:
        pass

    return False


async def enter_store_and_get_region(game_page, timeout_ms=15000):
    """
    进入 Cowbell Store -> Buy Cowbells，并返回尽可能收窄后的商店区域 locator。
    """
    
    store_btn = game_page.get_by_text(re.compile(r"^Cowbell Store$", re.I)).first
    buy_tab = game_page.get_by_text(re.compile(r"^Buy Cowbells$", re.I)).first

    await store_btn.wait_for(state="visible", timeout=timeout_ms)
    await store_btn.click()
    await game_page.wait_for_timeout(600)

    await buy_tab.wait_for(state="visible", timeout=timeout_ms)
    await buy_tab.click()
    await game_page.wait_for_timeout(800)

    # 优先找带 "Buy Cowbells" 文本的容器
    # 如果 DOM 结构不理想，再退回 page 级别
    region_candidates = [
        game_page.locator("section").filter(has_text=re.compile(r"Buy Cowbells", re.I)).first,
        game_page.locator("div").filter(has_text=re.compile(r"Buy Cowbells", re.I)).first,
        game_page.locator("main").first,
    ]

    for reg in region_candidates:
        try:
            if await reg.count() > 0 and await reg.first.is_visible():
                return reg.first
        except:
            pass

    return game_page


async def wait_for_100_package_in_region(region, timeout_sec=12):
    """
    只在商店区域里找 $100 包，而不是全页扫描。
    """
    candidates = [
        region.get_by_text(re.compile(r"11500\s+Cowbells", re.I)).first,
        region.get_by_text(re.compile(r"Price:\s*\$100", re.I)).first,
        region.get_by_role("button", name=re.compile(r"11500\s+Cowbells", re.I)).first,
        region.get_by_role("button", name=re.compile(r"Price:\s*\$100", re.I)).first,
    ]

    found = await first_visible_locator(candidates, timeout=timeout_sec)
    return found


async def dismiss_any_open_modal(game_page):
    """
    在商店页也顺手清一次遮挡层。
    """
    await maybe_close_post_login_dialog(game_page, timeout_ms=800)

await context.storage_state(path="mwidle_state.json", indexed_db=True)

In [ ]:
import asyncio
import pandas as pd

async def run_batch_headless(game_page, loops=100, quantity=99, delay_sec=0.5, stop_on_error=False):
    results = []

    for i in range(loops):
        print(f"\n===== iteration {i+1}/{loops} =====")

        try:
            checkout_page = await open_100_dollar_package(game_page)
            await set_quantity_on_checkout(checkout_page, quantity)

            mode = await submit_and_return_fast(
                checkout_page,
                game_page,
                refresh_game=False,
            )

            results.append({
                "iteration": i + 1,
                "ok": True,
                "mode": mode,
                "error": "",
            })

        except Exception as e:
            err = str(e)
            print("FAILED:", err)

            results.append({
                "iteration": i + 1,
                "ok": False,
                "mode": "",
                "error": err,
            })

            # 失败后尽量把游戏页拉回起点
            try:
                await game_page.goto(GAME_URL, wait_until="domcontentloaded")
                await game_page.wait_for_timeout(1500)
            except:
                pass

            if stop_on_error:
                break

        await asyncio.sleep(delay_sec)

    df = pd.DataFrame(results)
    df.to_csv("autopay_results.csv", index=False)
    return df

results = await run_batch_headless(
    game_page,
    loops=1,
    quantity=99,
    delay_sec=0.5,
    stop_on_error=True,
)

results

results = await run_batch_headless(
    game_page,
    loops=30000,
    quantity=99,
    delay_sec=0.5,
    stop_on_error=False,
)

results.tail()
await context.close()
await browser.close()
await p.stop()

In [ ]:
async def block_static_assets(route):
    rt = route.request.resource_type
    if rt in {"image", "font", "media"}:
        await route.abort()
    else:
        await route.continue_()

async def new_context_for_account(browser, account):
    context = await browser.new_context(
        storage_state=account["state_file"],
        service_workers="block",
        viewport={"width": 1280, "height": 900},
    )
    await context.route("**/*", block_static_assets)

    game_page = await context.new_page()
    await game_page.goto(account["game_url"], wait_until="domcontentloaded")
    return context, game_page

async def account_worker_separate_browser(playwright, account, loops=100, quantity=99, delay_sec=0.1):
    browser = await playwright.chromium.launch(
        headless=True,
    )

    rows = []
    context = None
    game_page = None
    pending_checkouts = []

    async def reset_context():
        nonlocal context, game_page, pending_checkouts
        if context is not None:
            try:
                await context.close()
            except:
                pass
        context, game_page = await new_context_for_account(browser, account)
        pending_checkouts = []

    try:
        await reset_context()

        REFRESH_EVERY = 50  # 比 20 更快；如果不稳再调回 30 或 20

        for i in range(1, loops + 1):
            checkout_page = None
            just_reloaded = False

            try:
                if i == 1 or (i % REFRESH_EVERY == 0):
                    await game_page.goto(account["game_url"], wait_until="domcontentloaded")
                    just_reloaded = True
                    await dismiss_offline_progress_modal(game_page, timeout_ms=2500)

                # 每轮先清理已完成的 tabs
                pending_checkouts, finished = await reap_finished_checkouts(pending_checkouts)

                # 如果挂起中的 tab 太多，就等一个空位
                pending_checkouts = await wait_for_checkout_slot(
                    pending_checkouts,
                    max_inflight=MAX_INFLIGHT_PER_ACCOUNT,
                )

                checkout_page = await open_100_dollar_package(
                    game_page,
                    game_url=None,
                    max_attempts=3,
                    just_reloaded=just_reloaded,
                )

                await set_quantity_on_checkout(checkout_page, quantity)

                pending_item = await submit_checkout_only_fast(checkout_page)
                pending_checkouts.append(pending_item)

                rows.append({
                    "account": account["name"],
                    "iteration": i,
                    "ok": True,
                    "mode": "submitted",
                    "error": "",
                    "pending_now": len(pending_checkouts),
                })

                if i % 20 == 0:
                    print(f'[{account["name"]}] iteration {i}/{loops}, inflight={len(pending_checkouts)}')

            except Exception as e:
                err = str(e)
                print(f'[{account["name"]}] FAILED: {err}')

                try:
                    if game_page is not None:
                        await game_page.screenshot(
                            path=str(DEBUG_DIR / f'{account["name"]}_game_fail_{i}.png'),
                            full_page=True
                        )
                except:
                    pass

                try:
                    if checkout_page is not None and not checkout_page.is_closed():
                        await checkout_page.screenshot(
                            path=str(DEBUG_DIR / f'{account["name"]}_checkout_fail_{i}.png'),
                            full_page=True
                        )
                except:
                    pass

                rows.append({
                    "account": account["name"],
                    "iteration": i,
                    "ok": False,
                    "mode": "",
                    "error": err,
                    "pending_now": len(pending_checkouts),
                })

                await reset_context()

            await asyncio.sleep(delay_sec)

        # 最后把剩余挂起的全部等完并关闭
        deadline = time.time() + 120
        while pending_checkouts and time.time() < deadline:
            pending_checkouts, finished = await reap_finished_checkouts(pending_checkouts)
            if pending_checkouts:
                await asyncio.sleep(0.5)

        # 兜底关闭残余 tabs
        for item in pending_checkouts:
            try:
                if not item["page"].is_closed():
                    await item["page"].close()
            except:
                pass

        return pd.DataFrame(rows)

    finally:
        if context is not None:
            try:
                await context.close()
            except:
                pass
        try:
            await browser.close()
        except:
            pass
            
# async def account_worker_separate_browser(playwright, account, loops=100, quantity=99, delay_sec=0.5):
#     browser = await playwright.chromium.launch(
#         channel="chromium",
#         headless=True,
#     )

#     rows = []
#     context = None
#     game_page = None

#     async def reset_context():
#         nonlocal context, game_page
#         if context is not None:
#             try:
#                 await context.close()
#             except:
#                 pass
#         context, game_page = await new_context_for_account(browser, account)

#     try:
#         await reset_context()

#         for i in range(1, loops + 1):
#             checkout_page = None
#             try:
#                 # print(f'[{account["name"]}] iteration {i}/{loops}')

#                 REFRESH_EVERY = 100

#                 if i == 1 or (i % REFRESH_EVERY == 0):
#                     await game_page.goto(account["game_url"], wait_until="domcontentloaded")
#                     await game_page.wait_for_timeout(600)
#                     await maybe_close_post_login_dialog(game_page, timeout_ms=2500)
#                     print(f'[{account["name"]}] iteration {i}/{loops}')
                
#                 checkout_page = await open_100_dollar_package(
#                     game_page,
#                     game_url=None,      # 不再每轮重复 goto
#                     max_attempts=3,
#                 )
#                 await set_quantity_on_checkout(checkout_page, quantity)

#                 mode = await submit_and_return_fast(
#                     checkout_page,
#                     game_page,
#                     refresh_game=False,
#                 )

#                 rows.append({
#                     "account": account["name"],
#                     "iteration": i,
#                     "ok": True,
#                     "mode": mode,
#                     "error": "",
#                 })

#             except Exception as e:
#                 err = str(e)
#                 print(f'[{account["name"]}] FAILED:', err)

#                 try:
#                     if game_page is not None:
#                         await game_page.screenshot(
#                             path=str(DEBUG_DIR / f'{account["name"]}_game_fail_{i}.png'),
#                             full_page=True
#                         )
#                 except:
#                     pass

#                 try:
#                     if checkout_page is not None:
#                         await checkout_page.screenshot(
#                             path=str(DEBUG_DIR / f'{account["name"]}_checkout_fail_{i}.png'),
#                             full_page=True
#                         )
#                 except:
#                     pass

#                 rows.append({
#                     "account": account["name"],
#                     "iteration": i,
#                     "ok": False,
#                     "mode": "",
#                     "error": err,
#                 })

#                 await reset_context()

#             await asyncio.sleep(delay_sec)

#         return pd.DataFrame(rows)

#     finally:
#         if context is not None:
#             try:
#                 await context.close()
#             except:
#                 pass
#         await browser.close()

async def run_parallel_two_browsers(accounts, loops=100, quantity=99, delay_sec=0.5):
    p = await async_playwright().start()
    try:
        dfs = await asyncio.gather(*[
            account_worker_separate_browser(
                playwright=p,
                account=account,
                loops=loops,
                quantity=quantity,
                delay_sec=delay_sec,
            )
            for account in accounts
        ])

        df = pd.concat(dfs, ignore_index=True)
        df = df.sort_values(["iteration", "account"]).reset_index(drop=True)
        df.to_csv("autopay_parallel_two_browsers.csv", index=False)
        return df

    finally:
        await p.stop()

async def wait_for_checkout_url(game_page, timeout_ms=4000):
    end = time.time() + timeout_ms / 1000
    while time.time() < end:
        try:
            url = await game_page.evaluate("window.__pw_last_open_url")
            if url and "checkout.stripe.com" in url:
                return url
        except:
            pass
        await asyncio.sleep(0.08)
    return None

In [ ]:
from playwright.async_api import async_playwright

async def save_state_from_existing_chrome(cdp_url, state_file):
    p = await async_playwright().start()
    browser = await p.chromium.connect_over_cdp(cdp_url)

    if not browser.contexts:
        await p.stop()
        raise RuntimeError(f"没有拿到 context: {cdp_url}")

    context = browser.contexts[0]

    await context.storage_state(
        path=state_file,
        indexed_db=True,
    )
    print(f"saved: {state_file}")

    # 不要 browser.close()，否则可能把你手动开的 Chrome 也关掉
    await p.stop()

In [ ]:
import subprocess
import os

def launch_instance(char_id, port, url):
    # 路径适配 (请根据你的系统修改)
    chrome_path = "/Applications/Google Chrome.app/Contents/MacOS/Google Chrome" 
    
    # 为每个角色创建独立的配置文件夹
    base_dir = os.path.abspath("./mw_profiles")
    profile_path = os.path.join(base_dir, f"profile_{char_id}")
    os.makedirs(profile_path, exist_ok=True)
    
    cmd = [
        chrome_path,
        f"--remote-debugging-port={port}",
        f"--user-data-dir={profile_path}",
        "--no-first-run",
        url
    ]
    
    subprocess.Popen(cmd)
    print(f"\n🚀 已启动窗口：角色 {char_id}")
    print(f"🔗 URL: {url}")
    print(f"📂 配置路径: {profile_path}")

# --- 执行流程 ---

# 1. 启动第一个
launch_instance("12345", "9222", "https://test.milkywayidle.com/game?characterId=12345")
input("👉 [操作指引] 请在窗口 12345 中完成登录，看到游戏界面后，回到这里按【回车】启动下一个...")

# 2. 启动第二个
launch_instance("67890", "9223", "https://test.milkywayidle.com/game?characterId=67890")
print("\n✅ 第二个窗口已启动！")
print("👉 请完成第二个号的登录。全部完成后，即可运行下方的保存 Cell。")

In [ ]:
from playwright.async_api import async_playwright

async def save_state_from_existing_chrome(cdp_url, state_file):
    p = await async_playwright().start()
    browser = await p.chromium.connect_over_cdp(cdp_url)

    if not browser.contexts:
        await p.stop()
        raise RuntimeError(f"没有拿到 context: {cdp_url}")

    context = browser.contexts[0]

    await context.storage_state(
        path=state_file,
        indexed_db=True,
    )
    print(f"saved: {state_file}")

    # 不要 browser.close()，否则可能把你手动开的 Chrome 也关掉
    await p.stop()


await save_state_from_existing_chrome(
    "http://localhost:9222",
    "mwidle_state_12345.json",
)

await save_state_from_existing_chrome(
    "http://localhost:9223",
    "mwidle_state_67890.json",
)

In [ ]:
ACCOUNTS = [
    {
        "name": "acct_12345",
        "game_url": "https://test.milkywayidle.com/game?characterId=12345",
        "state_file": "mwidle_state_12345.json",
    },
    {
        "name": "acct_67890",
        "game_url": "https://test.milkywayidle.com/game?characterId=67890",
        "state_file": "mwidle_state_67890.json",
    },
]

In [ ]:
results = await run_parallel_two_browsers(
    accounts=ACCOUNTS,
    loops=20000,
    quantity=99,
    delay_sec=0.1,
)
results

import inspect

print("dismiss_offline_progress_modal in globals:", "dismiss_offline_progress_modal" in globals())

if "open_100_dollar_package" in globals():
    print("open_100_dollar_package signature:", inspect.signature(open_100_dollar_package))

if "account_worker_separate_browser" in globals():
    print("account_worker_separate_browser signature:", inspect.signature(account_worker_separate_browser))

In [ ]:
import asyncio
import random
import re
import time
import gc
import pandas as pd
from pathlib import Path
from playwright.async_api import async_playwright

DEBUG_DIR = Path("autopay_debug")
DEBUG_DIR.mkdir(exist_ok=True)

SAVE_FAILURE_SCREENSHOTS = False
REFRESH_GAME_EVERY = 50
MAX_INFLIGHT_PER_ACCOUNT = 2
SUCCESS_PATTERNS = [
    re.compile(r"Payment successful", re.I),
    re.compile(r"Thanks|Thank you", re.I),
    re.compile(r"Complete|Completed", re.I),
    re.compile(r"Success", re.I),
]

def dprint(*args, **kwargs):
    pass


async def first_visible_locator(locators, timeout=5):
    end = time.time() + timeout
    while time.time() < end:
        for loc in locators:
            try:
                if await loc.count() > 0 and await loc.first.is_visible():
                    return loc.first
            except:
                pass
        await asyncio.sleep(0.15)
    return None


async def dismiss_offline_progress_modal(page, timeout_ms=2500):
    bg = page.locator('div[class*="OfflineProgressModal_background"]').first
    modal = page.locator('div[class*="OfflineProgressModal_modalContainer"]').first

    try:
        bg_visible = await bg.is_visible()
    except:
        bg_visible = False
    try:
        modal_visible = await modal.is_visible()
    except:
        modal_visible = False

    if not (bg_visible or modal_visible):
        return False

    btn = await first_visible_locator([
        modal.get_by_role("button", name=re.compile(r"close|关闭|ok|确定|collect|claim|x|×", re.I)),
        modal.get_by_text(re.compile(r"close|关闭|ok|确定|collect|claim|x|×", re.I)),
        modal.locator("button").first,
    ], timeout=0.8)

    if btn is not None:
        try:
            await btn.click()
        except:
            pass
    else:
        try:
            if bg_visible:
                await bg.click()
        except:
            pass
        try:
            await page.keyboard.press("Escape")
        except:
            pass

    end = asyncio.get_running_loop().time() + timeout_ms / 1000
    while asyncio.get_running_loop().time() < end:
        try:
            still_bg = await bg.is_visible()
        except:
            still_bg = False
        try:
            still_modal = await modal.is_visible()
        except:
            still_modal = False
        if not (still_bg or still_modal):
            return True
        await asyncio.sleep(0.1)

    return False


async def install_window_open_capture(page):
    await page.evaluate("""
    () => {
        if (window.__pw_open_capture_installed) return;

        window.__pw_open_capture_installed = true;
        window.__pw_last_open_url = null;

        const originalOpen = window.open.bind(window);
        window.__pw_original_open = originalOpen;

        window.open = function(url, ...args) {
            window.__pw_last_open_url = url;
            return null;
        };
    }
    """)


async def wait_for_checkout_url(game_page, timeout_ms=4000):
    end = time.time() + timeout_ms / 1000
    while time.time() < end:
        try:
            url = await game_page.evaluate("window.__pw_last_open_url")
            if url and "checkout.stripe.com" in url:
                return url
        except:
            pass
        await asyncio.sleep(0.08)
    return None


async def enter_store_and_get_region(game_page, timeout_ms=15000):
    store_btn = game_page.get_by_text(re.compile(r"^Cowbell Store$", re.I)).first
    buy_tab = game_page.get_by_text(re.compile(r"^Buy Cowbells$", re.I)).first

    await store_btn.wait_for(state="visible", timeout=timeout_ms)
    try:
        await store_btn.click(timeout=5000)
    except Exception as e:
        if "intercepts pointer events" in str(e):
            await dismiss_offline_progress_modal(game_page, timeout_ms=2500)
            await store_btn.click(timeout=5000)
        else:
            raise

    await buy_tab.wait_for(state="visible", timeout=timeout_ms)
    await buy_tab.click()

    region_candidates = [
        game_page.locator("section").filter(has_text=re.compile(r"Buy Cowbells", re.I)).first,
        game_page.locator("div").filter(has_text=re.compile(r"Buy Cowbells", re.I)).first,
        game_page.locator("main").first,
    ]

    for reg in region_candidates:
        try:
            if await reg.count() > 0 and await reg.first.is_visible():
                return reg.first
        except:
            pass

    return game_page


async def wait_for_100_package_in_region(region, timeout_sec=12):
    candidates = [
        region.get_by_text(re.compile(r"11500\s+Cowbells", re.I)).first,
        region.get_by_text(re.compile(r"Price:\s*\$100", re.I)).first,
        region.get_by_role("button", name=re.compile(r"11500\s+Cowbells", re.I)).first,
        region.get_by_role("button", name=re.compile(r"Price:\s*\$100", re.I)).first,
    ]
    return await first_visible_locator(candidates, timeout=timeout_sec)


async def new_checkout_worker_tab(context):
    pg = await context.new_page()
    await pg.goto("about:blank")
    return pg


async def open_100_dollar_package(game_page, game_url=None, max_attempts=3, just_reloaded=False):
    last_error = None

    for attempt in range(1, max_attempts + 1):
        try:
            if just_reloaded and attempt == 1:
                await game_page.wait_for_timeout(600)
                await dismiss_offline_progress_modal(game_page, timeout_ms=2500)

            region = await enter_store_and_get_region(game_page, timeout_ms=15000)

            package_el = await wait_for_100_package_in_region(region, timeout_sec=10 + attempt * 2)
            if package_el is None:
                raise RuntimeError("已进入商店区域，但 $100 包未出现。")

            await package_el.click()
            await game_page.wait_for_timeout(300)

            continue_btn = game_page.get_by_role("button", name=re.compile(r"^Continue$", re.I)).first
            modal_title = game_page.get_by_text(re.compile(r"Continue to Purchase", re.I)).first

            await modal_title.wait_for(state="visible", timeout=8000)
            await continue_btn.wait_for(state="visible", timeout=8000)

            await install_window_open_capture(game_page)
            await continue_btn.click()

            checkout_url = await wait_for_checkout_url(game_page, timeout_ms=4000)
            if not checkout_url or "checkout.stripe.com" not in checkout_url:
                raise RuntimeError(f"没有成功捕获 checkout URL: {checkout_url}")

            checkout_page = await new_checkout_worker_tab(game_page.context)
            await checkout_page.goto(checkout_url, wait_until="domcontentloaded")
            return checkout_page

        except Exception as e:
            last_error = e
            if "intercepts pointer events" in str(e):
                await dismiss_offline_progress_modal(game_page, timeout_ms=2500)
                just_reloaded = False
                continue

            if attempt < max_attempts:
                await game_page.wait_for_timeout(500 + 200 * attempt)

    raise RuntimeError(f"连 $100 包都没点到。最后错误: {last_error}")


async def first_visible(candidates, timeout_ms=1500):
    end = time.time() + timeout_ms / 1000
    while time.time() < end:
        for loc in candidates:
            try:
                cand = loc.first
                if await cand.count() > 0 and await cand.is_visible():
                    return cand
            except:
                pass
        await asyncio.sleep(0.12)
    return None


async def _clear_and_fill(locator, value: str):
    await locator.click()
    try:
        await locator.press("Meta+A")
    except:
        pass
    try:
        await locator.press("Control+A")
    except:
        pass
    await locator.fill(value)


async def _fill_if_needed(locator, value: str, skip_if_nonempty=True):
    if locator is None:
        return False
    try:
        current = await locator.input_value()
    except:
        current = ""
    if skip_if_nonempty and current.strip():
        return True
    await _clear_and_fill(locator, value)
    return True


TEST_CARD   = "4242424242424242"
TEST_EXP    = "1234"
TEST_CVC    = "123"
TEST_NAME   = "Test User"
TEST_POSTAL = "10001"


class FastManualFields:
    def __init__(self):
        self.pay_without_link = None
        self.submit = None
        self.card_number = None
        self.expiry = None
        self.cvc = None
        self.cardholder = None
        self.postal = None


async def _resolve_on_target(target, kind, timeout_ms=1500):
    if kind == "pay_without_link":
        return await first_visible([
            target.get_by_role("button", name=re.compile(r"Pay without Link", re.I)),
            target.get_by_text(re.compile(r"Pay without Link", re.I)),
        ], timeout_ms=timeout_ms)

    if kind == "submit":
        return await first_visible([
            target.locator("button[type='submit']"),
            target.get_by_role("button", name=re.compile(r"^(Pay|Complete|Confirm|Subscribe)", re.I)),
        ], timeout_ms=timeout_ms)

    if kind == "card_number":
        return await first_visible([
            target.get_by_label(re.compile(r"Card number", re.I)),
            target.get_by_placeholder(re.compile(r"1234 1234 1234 1234", re.I)),
            target.locator('input[autocomplete="cc-number"]'),
            target.locator('input[name*="cardnumber"]'),
        ], timeout_ms=timeout_ms)

    if kind == "expiry":
        return await first_visible([
            target.get_by_label(re.compile(r"MM\s*/\s*YY|Exp(?:iry|iration)", re.I)),
            target.get_by_placeholder(re.compile(r"MM\s*/\s*YY", re.I)),
            target.locator('input[autocomplete="cc-exp"]'),
            target.locator('input[name*="exp"]'),
        ], timeout_ms=timeout_ms)

    if kind == "cvc":
        return await first_visible([
            target.get_by_label(re.compile(r"CVC|CVV|Security code", re.I)),
            target.get_by_placeholder(re.compile(r"CVC|CVV", re.I)),
            target.locator('input[autocomplete="cc-csc"]'),
            target.locator('input[name*="cvc"], input[name*="cvv"]'),
        ], timeout_ms=timeout_ms)

    if kind == "cardholder":
        return await first_visible([
            target.get_by_label(re.compile(r"Cardholder name|Name on card", re.I)),
            target.get_by_placeholder(re.compile(r"Cardholder name|Name on card", re.I)),
            target.locator('input[autocomplete="cc-name"]'),
        ], timeout_ms=timeout_ms)

    if kind == "postal":
        return await first_visible([
            target.get_by_label(re.compile(r"ZIP|Postal code|Postcode", re.I)),
            target.get_by_placeholder(re.compile(r"ZIP|Postal code|Postcode", re.I)),
            target.locator('input[autocomplete="postal-code"]'),
        ], timeout_ms=timeout_ms)

    return None


async def resolve_manual_fields_fast(checkout_page):
    f = FastManualFields()

    f.pay_without_link = await _resolve_on_target(checkout_page, "pay_without_link", timeout_ms=1200)
    if f.pay_without_link is not None:
        await f.pay_without_link.click()
        await checkout_page.wait_for_timeout(600)

    f.card_number = await _resolve_on_target(checkout_page, "card_number", timeout_ms=5000)

    if f.card_number is not None:
        f.expiry     = await _resolve_on_target(checkout_page, "expiry", timeout_ms=800)
        f.cvc        = await _resolve_on_target(checkout_page, "cvc", timeout_ms=800)
        f.cardholder = await _resolve_on_target(checkout_page, "cardholder", timeout_ms=800)
        f.postal     = await _resolve_on_target(checkout_page, "postal", timeout_ms=800)

    f.submit = await _resolve_on_target(checkout_page, "submit", timeout_ms=1500)
    return f


async def fill_manual_fields_fast(fields):
    if fields.card_number is None:
        raise RuntimeError("快速路径下仍未定位到 card number。")

    await _clear_and_fill(fields.card_number, TEST_CARD)

    if fields.expiry is not None:
        await _clear_and_fill(fields.expiry, TEST_EXP)
    if fields.cvc is not None:
        await _clear_and_fill(fields.cvc, TEST_CVC)
    if fields.cardholder is not None:
        await _fill_if_needed(fields.cardholder, TEST_NAME, skip_if_nonempty=True)
    if fields.postal is not None:
        await _fill_if_needed(fields.postal, TEST_POSTAL, skip_if_nonempty=True)


async def submit_checkout_only_fast(checkout_page):
    fields = await resolve_manual_fields_fast(checkout_page)
    await fill_manual_fields_fast(fields)

    if fields.submit is None:
        raise RuntimeError("没有找到支付按钮。")

    await fields.submit.click()
    await checkout_page.wait_for_timeout(200)

    return {
        "page": checkout_page,
        "submitted_at": time.time(),
    }


async def checkout_is_done(page):
    if page.is_closed():
        return True
    try:
        url = page.url
    except:
        return True

    if "paymentSuccess" in url:
        return True
    if "checkout.stripe.com" not in url:
        return True

    for pat in SUCCESS_PATTERNS:
        try:
            loc = page.get_by_text(pat).first
            if await loc.count() > 0 and await loc.is_visible():
                return True
        except:
            pass
    return False


async def reap_finished_checkouts(pending_checkouts):
    still_pending = []
    finished = 0

    for item in pending_checkouts:
        page = item["page"]
        try:
            done = await checkout_is_done(page)
        except:
            done = True

        if done:
            try:
                if not page.is_closed():
                    await page.close()
            except:
                pass
            finished += 1
        else:
            still_pending.append(item)

    return still_pending, finished


async def wait_for_checkout_slot(pending_checkouts, max_inflight=MAX_INFLIGHT_PER_ACCOUNT):
    while len(pending_checkouts) >= max_inflight:
        pending_checkouts, finished = await reap_finished_checkouts(pending_checkouts)
        if finished == 0:
            await asyncio.sleep(0.3)
    return pending_checkouts


async def block_static_assets(route):
    rt = route.request.resource_type
    if rt in {"image", "font", "media"}:
        await route.abort()
    else:
        await route.continue_()


async def new_context_for_account(browser, account):
    context = await browser.new_context(
        storage_state=account["state_file"],
        service_workers="block",
        viewport={"width": 1280, "height": 900},
    )
    await context.route("**/*", block_static_assets)
    game_page = await context.new_page()
    await game_page.goto(account["game_url"], wait_until="domcontentloaded")
    return context, game_page


async def account_worker_separate_browser(playwright, account, loops=20000, quantity=99, delay_sec=0.1):
    browser = await playwright.chromium.launch(headless=True)

    rows = []
    context = None
    game_page = None
    pending_checkouts = []

    async def reset_context():
        nonlocal context, game_page, pending_checkouts
        if context is not None:
            try:
                await context.close()
            except:
                pass
        context, game_page = await new_context_for_account(browser, account)
        pending_checkouts = []

    try:
        await reset_context()

        for i in range(1, loops + 1):
            checkout_page = None
            just_reloaded = False

            try:
                if i == 1 or (i % REFRESH_GAME_EVERY == 0):
                    await game_page.goto(account["game_url"], wait_until="domcontentloaded")
                    just_reloaded = True
                    await dismiss_offline_progress_modal(game_page, timeout_ms=2500)

                pending_checkouts, _ = await reap_finished_checkouts(pending_checkouts)
                pending_checkouts = await wait_for_checkout_slot(
                    pending_checkouts,
                    max_inflight=MAX_INFLIGHT_PER_ACCOUNT,
                )

                checkout_page = await open_100_dollar_package(
                    game_page,
                    game_url=None,
                    max_attempts=3,
                    just_reloaded=just_reloaded,
                )

                await set_quantity_on_checkout(checkout_page, quantity)

                pending_item = await submit_checkout_only_fast(checkout_page)
                pending_checkouts.append(pending_item)

                rows.append({
                    "account": account["name"],
                    "iteration": i,
                    "ok": True,
                    "mode": "submitted",
                    "error": "",
                    "pending_now": len(pending_checkouts),
                })

                if i % 20 == 0:
                    print(f'[{account["name"]}] iteration {i}/{loops}')

            except Exception as e:
                err = str(e)
                print(f'[{account["name"]}] FAILED: {err}')

                if SAVE_FAILURE_SCREENSHOTS:
                    try:
                        if game_page is not None:
                            await game_page.screenshot(
                                path=str(DEBUG_DIR / f'{account["name"]}_game_fail_{i}.png'),
                                full_page=True
                            )
                    except:
                        pass

                    try:
                        if checkout_page is not None and not checkout_page.is_closed():
                            await checkout_page.screenshot(
                                path=str(DEBUG_DIR / f'{account["name"]}_checkout_fail_{i}.png'),
                                full_page=True
                            )
                    except:
                        pass

                rows.append({
                    "account": account["name"],
                    "iteration": i,
                    "ok": False,
                    "mode": "",
                    "error": err,
                    "pending_now": len(pending_checkouts),
                })

                await reset_context()

            await asyncio.sleep(delay_sec)

        deadline = time.time() + 120
        while pending_checkouts and time.time() < deadline:
            pending_checkouts, _ = await reap_finished_checkouts(pending_checkouts)
            if pending_checkouts:
                await asyncio.sleep(0.5)

        for item in pending_checkouts:
            try:
                if not item["page"].is_closed():
                    await item["page"].close()
            except:
                pass

        return pd.DataFrame(rows)

    finally:
        try:
            if context is not None:
                await context.close()
        except:
            pass
        try:
            await browser.close()
        except:
            pass
        gc.collect()


async def run_parallel_two_browsers(accounts, loops=20000, quantity=99, delay_sec=0.1):
    p = await async_playwright().start()
    try:
        dfs = await asyncio.gather(*[
            account_worker_separate_browser(
                playwright=p,
                account=account,
                loops=loops,
                quantity=quantity,
                delay_sec=delay_sec,
            )
            for account in accounts
        ], return_exceptions=True)

        final_dfs = []
        for account, res in zip(accounts, dfs):
            if isinstance(res, Exception):
                print(f'[{account["name"]}] worker crashed: {res}')
                final_dfs.append(pd.DataFrame([{
                    "account": account["name"],
                    "iteration": None,
                    "ok": False,
                    "mode": "",
                    "error": f"worker crashed: {res}",
                }]))
            else:
                final_dfs.append(res)

        df = pd.concat(final_dfs, ignore_index=True)
        df.to_csv("autopay_parallel_two_browsers.csv", index=False)
        return df

    finally:
        try:
            await p.stop()
        except:
            pass
        gc.collect()

In [ ]:
import inspect

print("dismiss_offline_progress_modal in globals:", "dismiss_offline_progress_modal" in globals())
print("open_100_dollar_package signature:", inspect.signature(open_100_dollar_package))

In [ ]:
results = await run_parallel_two_browsers(
    accounts=ACCOUNTS,
    loops=20000,
    quantity=99,
    delay_sec=0.1,
)